# PixelRAG Embedding Server

## How to use
1. Make sure you have selected **Runtime → Change runtime type → T4 GPU** (or better).
2. Run **all cells** (Runtime → Run all).
3. Wait for the final cell to print a line like:
   ```
   ✅  Embedding server is live at: https://xxxx-xx-xx-xxx-xxx.ngrok-free.app
   ```
4. **Copy that URL** and paste it into the local app's `EMBED_API_URL` setting (the sidebar field in the Streamlit GUI, or the `.env` file).

## Important notes
- **ngrok account required (free):** Sign up at https://ngrok.com, get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken, and paste it into the cell below.
- **The URL changes every time you restart the notebook.** If Colab disconnects (idle timeout ~90 min, or the 12-hour session limit), you must re-run all cells and copy the new URL into the local app.
- The model (`Qwen/Qwen3-VL-Embedding-2B`) is loaded once at startup in fp16 — it needs roughly 5–6 GB of GPU VRAM (fits on a free T4).

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers accelerate torch fastapi uvicorn pyngrok Pillow einops qwen-vl-utils

In [ ]:
# Cell 2 — Set your ngrok authtoken (get it from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"  # <-- replace this

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTHTOKEN

In [ ]:
# Cell 3 — Load tomaarsen/Qwen3-VL-Embedding-2B-vdr on GPU (fp16, loaded once)
import torch
from transformers import AutoProcessor, AutoModel
from PIL import Image
import io, base64, json

MODEL_ID = "tomaarsen/Qwen3-VL-Embedding-2B-vdr"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model (fp16)...")
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto",
)
model.eval()
print("Model loaded.")

In [ ]:
# Cell 4 — Embedding helper functions (mean pooling + visual instruction prefix)
import torch.nn.functional as F

# Instruction prefixes — VDR models benefit from task-specific prompts
IMAGE_INSTRUCTION = "Represent this document image for retrieval, including all visible text, charts, figures, tables, and diagrams."
TEXT_INSTRUCTION  = "Query: "


def _mean_pool(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Weighted mean pooling — ignores padding tokens."""
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return F.normalize(summed / counts, p=2, dim=-1)


def embed_image_pil(pil_image: Image.Image) -> list:
    """Embed a PIL image using Qwen3-VL-Embedding-2B-vdr with visual instruction prefix."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text",  "text": IMAGE_INSTRUCTION},
            ],
        }
    ]
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_prompt],
        images=[pil_image],
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1]
        embedding = _mean_pool(last_hidden, inputs["attention_mask"])

    return embedding[0].float().cpu().tolist()


def embed_text_query(text: str) -> list:
    """Embed a text query with the VDR query prefix."""
    prefixed = TEXT_INSTRUCTION + text
    messages = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prefixed}],
        }
    ]
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_prompt],
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1]
        embedding = _mean_pool(last_hidden, inputs["attention_mask"])

    return embedding[0].float().cpu().tolist()


print("Embedding helpers ready.")

In [ ]:
# Cell 5 — Define FastAPI application
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List

app = FastAPI(title="PixelRAG Embedding Server")


class ImageRequest(BaseModel):
    image_b64: str  # base64-encoded PNG/JPEG


class TextRequest(BaseModel):
    text: str


class EmbedResponse(BaseModel):
    embedding: List[float]
    dim: int


@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE}


@app.post("/embed_image", response_model=EmbedResponse)
def embed_image_endpoint(req: ImageRequest):
    try:
        img_bytes = base64.b64decode(req.image_b64)
        pil_image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Cannot decode image: {e}")
    try:
        vec = embed_image_pil(pil_image)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Embedding failed: {e}")
    return EmbedResponse(embedding=vec, dim=len(vec))


@app.post("/embed_text", response_model=EmbedResponse)
def embed_text_endpoint(req: TextRequest):
    if not req.text.strip():
        raise HTTPException(status_code=400, detail="text must not be empty")
    try:
        vec = embed_text_query(req.text)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Embedding failed: {e}")
    return EmbedResponse(embedding=vec, dim=len(vec))


print("FastAPI app defined.")

In [ ]:
# Cell 6 — Start uvicorn in the background and open ngrok tunnel
import threading, time, requests as req_lib
import uvicorn

PORT = 8000


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give uvicorn a moment to start
time.sleep(3)

# Verify locally before tunnelling
try:
    r = req_lib.get(f"http://localhost:{PORT}/health", timeout=5)
    print("Local health check:", r.json())
except Exception as e:
    print(f"⚠️  Local health check failed: {e} — server may still be starting")

# Open ngrok tunnel
tunnel = ngrok.connect(PORT, "http")
PUBLIC_URL = tunnel.public_url

print()
print("=" * 60)
print(f"✅  Embedding server is live at: {PUBLIC_URL}")
print()
print("Copy the URL above and paste it into the local Streamlit app's")
print("EMBED_API_URL setting (sidebar field or .env file).")
print()
print("REMINDER: This URL changes every time the notebook restarts.")
print("If Colab disconnects, re-run all cells and copy the new URL.")
print("=" * 60)

In [ ]:
# Cell 7 — Optional smoke test (run after the server is live)
import requests as req_lib, base64, json
from PIL import Image, ImageDraw
import io

# Create a tiny test image
img = Image.new("RGB", (64, 64), color=(100, 150, 200))
draw = ImageDraw.Draw(img)
draw.text((5, 20), "test", fill=(255, 255, 255))
buf = io.BytesIO()
img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode()

# Hit /embed_image
r = req_lib.post(f"{PUBLIC_URL}/embed_image", json={"image_b64": b64}, timeout=30)
print("embed_image status:", r.status_code)
data = r.json()
print(f"  dim={data['dim']}, first 5 values={data['embedding'][:5]}")

# Hit /embed_text
r2 = req_lib.post(f"{PUBLIC_URL}/embed_text", json={"text": "a blue test image"}, timeout=30)
print("embed_text status:", r2.status_code)
data2 = r2.json()
print(f"  dim={data2['dim']}, first 5 values={data2['embedding'][:5]}")

print("\n✅  Smoke test passed — server is ready.")